In [1]:
!pip install pydantic

In [2]:
"""
API with JSON Validation using Pydantic 

Key concepts covered:
- Defining data models with type hints
- Setting default values for optional fields
- Custom validation rules using @validator decorator
- Error handling for invalid data
"""

from pydantic import BaseModel, Field, validator
from typing import Optional
import json

print("=" * 50)
print("PYDANTIC BASICS")
print("=" * 50)


class SimpleProduct(BaseModel):
    """
    A simple product model for validation.
    
    This model defines the expected structure for product data.
    Pydantic will automatically:
    - Check that 'name' is a string
    - Check that 'price' is a float
    - Check that 'quantity' is an integer (defaults to 1 if not provided)
    - Run custom validators for business rules
    
    Attributes:
        name (str): The product name. Required, no default.
        price (float): The product price. Must be positive.
        quantity (int): How many units. Defaults to 1. Must be positive.
    """

    name: str
    price: float
    quantity: int = 1  # Default value — if no quantity is passed, it's set to 1

    @validator('price')
    def price_must_be_positive(cls, v):
        """
        Custom validator for price field.
        
        Pydantic calls this automatically when creating a SimpleProduct.
        'cls' = the class itself (like a classmethod)
        'v' = the value being validated
        
        Returns the value if valid, raises ValueError if not.
        """
        if v <= 0:
            raise ValueError('Price must be positive')
        return v

    @validator('quantity')
    def quantity_must_be_positive(cls, v):
        """
        Custom validator for quantity field.
        
        Same pattern as price — ensures we never accept zero or
        negative quantities, which would break order processing.
        """
        if v <= 0:
            raise ValueError('Quantity must be positive')
        return v


# --- Test 1: Valid data ---
# This should pass all validation checks and create a product object
print("\n1. Valid data:")
try:
    product1 = SimpleProduct(name="Widget", price=10.99, quantity=5)
    print(f"  ✓ Valid: {product1.name} - ${product1.price}")
except Exception as e:
    print(f"  ✗ Error: {e}")

# --- Test 2: Invalid data ---
# This should FAIL because price is negative, triggering our @validator
print("\n2. Invalid data (negative price):")
try:
    product2 = SimpleProduct(name="Widget", price=-10.99)
except Exception as e:
    # We EXPECT this error — it means validation is working correctly
    print(f"  ✗ Validation error (expected): {e}")

print("\n✓ Pydantic basics working!")

PYDANTIC BASICS

1. Valid data:
  ✓ Valid: Widget - $10.99

2. Invalid data (negative price):
  ✗ Validation error (expected): 1 validation error for SimpleProduct
price
  Value error, Price must be positive [type=value_error, input_value=-10.99, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

✓ Pydantic basics working!


C:\Users\eugnm\AppData\Local\Temp\ipykernel_23228\3738636780.py:41: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator('price')
C:\Users\eugnm\AppData\Local\Temp\ipykernel_23228\3738636780.py:56: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator('quantity')


In [4]:
"""
Step 2: Product Data Models for Listing Generator
===================================================
These Pydantic models validate product data before it gets sent to the
OpenAI API for listing generation. This prevents bad data from wasting
API calls (and money).

The models mirror the fields used in the previous lab's
create_product_listing_prompt() function.
"""

from pydantic import BaseModel, Field, validator
from typing import Optional, List


class ProductRequest(BaseModel):
    """
    Validates incoming product data before generating a listing.

    This model ensures every product has the required fields and that
    all values make business sense before we spend tokens on an API call.

    Attributes:
        name (str): Product display name. Must be at least 2 characters.
        price (float): Product price in USD. Must be between 0.01 and 99999.
        category (str): Main category (e.g. "Apparel", "Accessories").
        image_path (str): Path to the product image file.
        additional_info (str, optional): Extra details like "Noise cancelling, 30hr battery".
        target_audience (str, optional): Who the product is for. Defaults to "general".
    """

    name: str = Field(..., min_length=2, max_length=200, description="Product display name")
    price: float = Field(..., gt=0, le=99999, description="Price in USD")
    category: str = Field(..., min_length=2, description="Product category")
    image_path: str = Field(..., description="Path to product image")
    additional_info: Optional[str] = Field(None, max_length=500, description="Extra product details")
    target_audience: Optional[str] = Field("general", description="Target customer segment")

    @validator('name')
    def name_must_not_be_empty_spaces(cls, v):
        """Reject names that are just whitespace."""
        if not v.strip():
            raise ValueError('Product name cannot be blank')
        return v.strip()

    @validator('category')
    def category_must_be_valid(cls, v):
        """
        Ensure category matches the dataset categories.
        These come from the fashion-product-images dataset used in LAB4.
        """
        valid_categories = [
            "Apparel", "Accessories", "Footwear",
            "Personal Care", "Free Items", "Electronics"
        ]
        if v not in valid_categories:
            raise ValueError(f'Category must be one of: {valid_categories}')
        return v

    @validator('image_path')
    def image_must_be_valid_format(cls, v):
        """Only accept common image formats."""
        valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
        if not v.lower().endswith(valid_extensions):
            raise ValueError(f'Image must be one of: {valid_extensions}')
        return v


class ProductListing(BaseModel):
    """
    Validates the JSON response coming back from the OpenAI API.

    After GPT-4.1 generates a listing, we parse it through this model
    to make sure the output has all required fields and meets quality standards.

    Attributes:
        title (str): SEO-friendly product title, max 60 chars.
        description (str): Full product description, 100-1000 chars.
        features (list): 3-7 bullet point features.
        keywords (str): Comma-separated SEO keywords.
    """

    title: str = Field(..., max_length=60, description="SEO-friendly title")
    description: str = Field(..., min_length=100, max_length=1000, description="Product description")
    features: List[str] = Field(..., min_items=3, max_items=7, description="Key features list")
    keywords: str = Field(..., description="Comma-separated SEO keywords")

    @validator('title')
    def title_not_empty(cls, v):
        """Reject blank titles."""
        if not v.strip():
            raise ValueError('Title cannot be blank')
        return v.strip()

    @validator('keywords')
    def keywords_must_have_enough(cls, v):
        """Ensure we get at least 5 keywords for SEO value."""
        keyword_list = [k.strip() for k in v.split(',') if k.strip()]
        if len(keyword_list) < 5:
            raise ValueError(f'Need at least 5 keywords, got {len(keyword_list)}')
        return v


# ============================================================
# TESTING - Verify all validation rules work
# ============================================================

print("=" * 50)
print("TESTING ProductRequest VALIDATION")
print("=" * 50)

# --- Test 1: Valid product ---
print("\n1. Valid product:")
try:
    product = ProductRequest(
        name="Turtle Check Men Navy Blue Shirt",
        price=29.99,
        category="Apparel",
        image_path="product_images/0.jpg",
        additional_info="Cotton, regular fit"
    )
    print(f"  ✓ {product.name} - ${product.price} [{product.category}]")
except Exception as e:
    print(f"  ✗ {e}")

# --- Test 2: Negative price ---
print("\n2. Negative price:")
try:
    bad_product = ProductRequest(
        name="Test Shirt",
        price=-10.00,
        category="Apparel",
        image_path="product_images/0.jpg"
    )
except Exception as e:
    print(f"  ✓ Caught error (expected): price must be > 0")

# --- Test 3: Invalid category ---
print("\n3. Invalid category:")
try:
    bad_product = ProductRequest(
        name="Test Shirt",
        price=29.99,
        category="FakeCategory",
        image_path="product_images/0.jpg"
    )
except Exception as e:
    print(f"  ✓ Caught error (expected): invalid category")

# --- Test 4: Bad image format ---
print("\n4. Bad image format:")
try:
    bad_product = ProductRequest(
        name="Test Shirt",
        price=29.99,
        category="Apparel",
        image_path="product_images/0.bmp"
    )
except Exception as e:
    print(f"  ✓ Caught error (expected): invalid image format")

# --- Test 5: Empty name ---
print("\n5. Empty name:")
try:
    bad_product = ProductRequest(
        name="   ",
        price=29.99,
        category="Apparel",
        image_path="product_images/0.jpg"
    )
except Exception as e:
    print(f"  ✓ Caught error (expected): blank name")

# --- Test 6: Missing required field ---
print("\n6. Missing required field (no price):")
try:
    bad_product = ProductRequest(
        name="Test Shirt",
        category="Apparel",
        image_path="product_images/0.jpg"
    )
except Exception as e:
    print(f"  ✓ Caught error (expected): price is required")


print("\n" + "=" * 50)
print("TESTING ProductListing VALIDATION (API response)")
print("=" * 50)

# --- Test 7: Valid listing ---
print("\n7. Valid listing response:")
try:
    listing = ProductListing(
        title="Navy Blue Check Shirt - Modern Casual Style",
        description="Elevate your wardrobe with this classic navy blue checkered shirt. " * 5,
        features=[
            "Premium cotton fabric",
            "Classic checkered pattern",
            "Regular fit",
            "Spread collar",
            "Machine washable"
        ],
        keywords="men's shirt, navy blue, check pattern, casual, cotton, formal, slim fit"
    )
    print(f"  ✓ Title: {listing.title}")
    print(f"  ✓ Features: {len(listing.features)} items")
except Exception as e:
    print(f"  ✗ {e}")

# --- Test 8: Too few keywords ---
print("\n8. Too few keywords:")
try:
    bad_listing = ProductListing(
        title="Test Shirt",
        description="A nice shirt. " * 20,
        features=["Feature 1", "Feature 2", "Feature 3"],
        keywords="shirt, blue"
    )
except Exception as e:
    print(f"  ✓ Caught error (expected): not enough keywords")

print("\n✓ All validation tests complete!")

TESTING ProductRequest VALIDATION

1. Valid product:
  ✓ Turtle Check Men Navy Blue Shirt - $29.99 [Apparel]

2. Negative price:
  ✓ Caught error (expected): price must be > 0

3. Invalid category:
  ✓ Caught error (expected): invalid category

4. Bad image format:
  ✓ Caught error (expected): invalid image format

5. Empty name:
  ✓ Caught error (expected): blank name

6. Missing required field (no price):
  ✓ Caught error (expected): price is required

TESTING ProductListing VALIDATION (API response)

7. Valid listing response:
  ✓ Title: Navy Blue Check Shirt - Modern Casual Style
  ✓ Features: 5 items

8. Too few keywords:
  ✓ Caught error (expected): not enough keywords

✓ All validation tests complete!


C:\Users\eugnm\AppData\Local\Temp\ipykernel_9332\1263811765.py:39: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator('name')
C:\Users\eugnm\AppData\Local\Temp\ipykernel_9332\1263811765.py:46: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator('category')
C:\Users\eugnm\AppData\Local\Temp\ipykernel_9332\1263811765.py:60: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should m

In [6]:
"""
Step 3: Validating JSON Input from Files
==========================================
This step creates test JSON files (valid and invalid), then builds
a validation function that loads and validates them using the Pydantic
models from Step 2.

Why this matters: In production, product data often comes from files,
databases, or API requests. Validating before processing prevents
bad data from wasting API calls and money.
"""

import json
import os

# ============================================================
# PART 1: Generate example JSON files for testing
# ============================================================

print("=" * 50)
print("CREATING TEST JSON FILES")
print("=" * 50)

# Create a test_data folder to keep things tidy
os.makedirs("test_data", exist_ok=True)

# --- Valid product data ---
valid_products = [
    {
        "name": "Turtle Check Men Navy Blue Shirt",
        "price": 29.99,
        "category": "Apparel",
        "image_path": "product_images/0.jpg",
        "additional_info": "Cotton, regular fit"
    },
    {
        "name": "Peter England Blue Party Jeans",
        "price": 49.99,
        "category": "Apparel",
        "image_path": "product_images/1.jpg",
        "additional_info": "Slim fit, stretchable denim"
    },
    {
        "name": "Titan Women Silver Watch",
        "price": 89.99,
        "category": "Accessories",
        "image_path": "product_images/2.jpg",
        "target_audience": "women"
    }
]

# --- Invalid product data (each has a different problem) ---
invalid_products = [
    {
        # Problem: negative price
        "name": "Bad Price Shirt",
        "price": -15.00,
        "category": "Apparel",
        "image_path": "product_images/0.jpg"
    },
    {
        # Problem: missing required field (no name)
        "price": 29.99,
        "category": "Apparel",
        "image_path": "product_images/0.jpg"
    },
    {
        # Problem: invalid category
        "name": "Mystery Item",
        "price": 19.99,
        "category": "MadeUpCategory",
        "image_path": "product_images/0.jpg"
    },
    {
        # Problem: wrong image format
        "name": "Bad Image Product",
        "price": 39.99,
        "category": "Footwear",
        "image_path": "product_images/0.gif"
    },
    {
        # Problem: blank name (just spaces)
        "name": "   ",
        "price": 25.00,
        "category": "Apparel",
        "image_path": "product_images/0.jpg"
    }
]

# Save both to JSON files
with open("test_data/valid_products.json", "w") as f:
    json.dump(valid_products, f, indent=4)

with open("test_data/invalid_products.json", "w") as f:
    json.dump(invalid_products, f, indent=4)

# Also create a single valid product file for simple testing
with open("test_data/single_product.json", "w") as f:
    json.dump(valid_products[0], f, indent=4)

# And a completely broken JSON file (malformed syntax)
with open("test_data/broken.json", "w") as f:
    f.write('{"name": "Missing bracket", "price": 29.99')

print("✓ Created test_data/valid_products.json   (3 valid products)")
print("✓ Created test_data/invalid_products.json  (5 invalid products)")
print("✓ Created test_data/single_product.json    (1 valid product)")
print("✓ Created test_data/broken.json            (malformed JSON)")


# ============================================================
# PART 2: Validation function
# ============================================================

print("\n" + "=" * 50)
print("VALIDATION FUNCTION")
print("=" * 50)


def validate_product_file(filepath):
    """
    Load a JSON file and validate its contents using ProductRequest.

    Handles three types of problems:
    1. File not found
    2. Malformed JSON (syntax errors)
    3. Invalid data (fails Pydantic validation)

    Args:
        filepath (str): Path to the JSON file

    Returns:
        dict with:
            - 'valid' (list): Successfully validated ProductRequest objects
            - 'errors' (list): Details about each failed product
            - 'summary' (str): Quick overview of results
    """

    results = {
        "valid": [],
        "errors": [],
        "summary": ""
    }

    # --- Check 1: Does the file exist? ---
    if not os.path.exists(filepath):
        results["errors"].append({
            "type": "FileNotFound",
            "message": f"File not found: {filepath}"
        })
        results["summary"] = "Failed: file not found"
        return results

    # --- Check 2: Is it valid JSON? ---
    try:
        with open(filepath, "r") as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        results["errors"].append({
            "type": "JSONError",
            "message": f"Malformed JSON: {str(e)}"
        })
        results["summary"] = "Failed: invalid JSON syntax"
        return results

    # --- Check 3: Validate with Pydantic ---
    # Handle both single product (dict) and multiple products (list)
    if isinstance(data, dict):
        data = [data]

    for i, product_data in enumerate(data):
        try:
            validated = ProductRequest(**product_data)
            results["valid"].append(validated)
        except Exception as e:
            results["errors"].append({
                "type": "ValidationError",
                "index": i,
                "data": product_data,
                "message": str(e)
            })

    # --- Build summary ---
    total = len(data)
    valid_count = len(results["valid"])
    error_count = len(results["errors"])
    results["summary"] = f"{valid_count}/{total} valid, {error_count} errors"

    return results


def print_results(results, label=""):
    """Pretty-print validation results."""
    if label:
        print(f"\n--- {label} ---")
    print(f"  Summary: {results['summary']}")

    if results["valid"]:
        print(f"  Valid products:")
        for p in results["valid"]:
            print(f"    ✓ {p.name} - ${p.price} [{p.category}]")

    if results["errors"]:
        print(f"  Errors:")
        for err in results["errors"]:
            if err["type"] == "ValidationError":
                name = err["data"].get("name", "MISSING NAME")
                print(f"    ✗ Product '{name}' (index {err['index']}): {err['type']}")
            else:
                print(f"    ✗ {err['type']}: {err['message']}")


print("✓ validate_product_file() defined")
print("✓ print_results() defined")


# ============================================================
# PART 3: Run validation on all test files
# ============================================================

print("\n" + "=" * 50)
print("RUNNING VALIDATION TESTS")
print("=" * 50)

# Test 1: Valid products file
results = validate_product_file("test_data/valid_products.json")
print_results(results, "valid_products.json")

# Test 2: Invalid products file
results = validate_product_file("test_data/invalid_products.json")
print_results(results, "invalid_products.json")

# Test 3: Single product file
results = validate_product_file("test_data/single_product.json")
print_results(results, "single_product.json")

# Test 4: Broken JSON file
results = validate_product_file("test_data/broken.json")
print_results(results, "broken.json")

# Test 5: File that doesn't exist
results = validate_product_file("test_data/nope.json")
print_results(results, "nope.json (missing file)")

print("\n✓ All validation tests complete!")

CREATING TEST JSON FILES
✓ Created test_data/valid_products.json   (3 valid products)
✓ Created test_data/invalid_products.json  (5 invalid products)
✓ Created test_data/single_product.json    (1 valid product)
✓ Created test_data/broken.json            (malformed JSON)

VALIDATION FUNCTION
✓ validate_product_file() defined
✓ print_results() defined

RUNNING VALIDATION TESTS

--- valid_products.json ---
  Summary: 3/3 valid, 0 errors
  Valid products:
    ✓ Turtle Check Men Navy Blue Shirt - $29.99 [Apparel]
    ✓ Peter England Blue Party Jeans - $49.99 [Apparel]
    ✓ Titan Women Silver Watch - $89.99 [Accessories]

--- invalid_products.json ---
  Summary: 0/5 valid, 5 errors
  Errors:
    ✗ Product 'Bad Price Shirt' (index 0): ValidationError
    ✗ Product 'MISSING NAME' (index 1): ValidationError
    ✗ Product 'Mystery Item' (index 2): ValidationError
    ✗ Product 'Bad Image Product' (index 3): ValidationError
    ✗ Product '   ' (index 4): ValidationError

--- single_product.json 

In [8]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
import json

class ProductRequest(BaseModel):
    name:       str   = Field(..., min_length=1)
    price:      float = Field(..., gt=0)
    category:   str   = Field(..., min_length=2)
    image_path: str   = Field(..., pattern=r".*\.(jpg|jpeg|png)$")

    model_config = {"extra": "ignore"}

In [9]:
# ============================================================
# STEP 4: JSON PARSING VALIDATION
# Validate JSON format BEFORE Pydantic validation
# ============================================================

print("\n" + "=" * 50)
print("STEP 4: JSON PARSING VALIDATION")
print("=" * 50)

def safe_parse_json(raw_input):
    """
    Safely parse a JSON string before passing to Pydantic.
    
    Why this matters: If JSON is malformed, Pydantic never
    even gets to run. We need to catch syntax errors first
    and give clear messages about what went wrong.
    
    Returns:
        (data, error) — one will always be None
    """
    try:
        data = json.loads(raw_input)
        return data, None
    except json.JSONDecodeError as e:
        error_msg = (
            f"JSON syntax error at line {e.lineno}, column {e.colno}: "
            f"{e.msg}. Check for missing brackets, quotes, or commas."
        )
        return None, error_msg


def validate_product_from_string(raw_json_string):
    """
    Full pipeline: parse JSON first, then validate with Pydantic.
    
    Step 1 — JSON parsing (catches syntax errors)
    Step 2 — Pydantic validation (catches business logic errors)
    """
    # Step 1: Parse JSON
    data, parse_error = safe_parse_json(raw_json_string)
    if parse_error:
        return {
            "success": False,
            "stage": "JSON_PARSING",
            "error": parse_error,
            "product": None
        }

    # Step 2: Pydantic validation
    try:
        product = ProductRequest(**data)
        return {
            "success": True,
            "stage": "VALIDATED",
            "error": None,
            "product": product
        }
    except ValidationError as e:
        error_details = []
        for err in e.errors():
            field = " -> ".join(str(x) for x in err["loc"])
            error_details.append(f"  Field '{field}': {err['msg']}")
        return {
            "success": False,
            "stage": "PYDANTIC_VALIDATION",
            "error": "\n".join(error_details),
            "product": None
        }


# Test cases
test_inputs = [
    ('{"name": "Blue Shirt", "price": 29.99, "category": "Apparel", "image_path": "product_images/0.jpg"}',
     "Valid JSON + valid data"),
    ('{"name": "Missing bracket", "price": 29.99',
     "Malformed JSON"),
    ('{"name": "Bad Price", "price": -5.00, "category": "Apparel", "image_path": "product_images/0.jpg"}',
     "Valid JSON + invalid data"),
    ('{not valid json at all!!!}',
     "Completely broken JSON"),
]

for raw_input, label in test_inputs:
    result = validate_product_from_string(raw_input)
    status = "✓" if result["success"] else "✗"
    print(f"\n{status} [{label}]")
    print(f"  Stage: {result['stage']}")
    if result["success"]:
        print(f"  Product: {result['product'].name} — ${result['product'].price}")
    else:
        print(f"  Error: {result['error']}")

print("\n✓ Step 4 complete")


STEP 4: JSON PARSING VALIDATION

✓ [Valid JSON + valid data]
  Stage: VALIDATED
  Product: Blue Shirt — $29.99

✗ [Malformed JSON]
  Stage: JSON_PARSING
  Error: JSON syntax error at line 1, column 43: Expecting ',' delimiter. Check for missing brackets, quotes, or commas.

✗ [Valid JSON + invalid data]
  Stage: PYDANTIC_VALIDATION
  Error:   Field 'price': Input should be greater than 0

✗ [Completely broken JSON]
  Stage: JSON_PARSING
  Error: JSON syntax error at line 1, column 2: Expecting property name enclosed in double quotes. Check for missing brackets, quotes, or commas.

✓ Step 4 complete


In [6]:
from pydantic import BaseModel, Field, field_validator, model_validator
from pydantic import ValidationError
from typing import Optional

In [7]:
# ============================================================
# STEP 5: CONTROLLING VALIDATION STRICTNESS
# ============================================================

print("\n" + "=" * 50)
print("STEP 5: VALIDATION STRICTNESS CONTROL")
print("=" * 50)


class ProductRequestStrict(BaseModel):
    """
    Strict validation — for production use.
    Extra fields are FORBIDDEN to catch typos in field names.
    Tight constraints on all values.
    """
    name:       str   = Field(..., min_length=3, max_length=100)
    price:      float = Field(..., gt=0, lt=10000)
    category:   str   = Field(..., min_length=2)
    image_path: str   = Field(..., pattern=r".*\.(jpg|jpeg|png)$")

    @validator("name")
    def name_not_blank(cls, v):
        if not v.strip():
            raise ValueError("Name cannot be blank")
        return v.strip()

    @validator("category")
    def valid_category(cls, v):
        allowed = {"Apparel", "Footwear", "Accessories",
                   "Personal Care", "Sporting Goods", "Home Decor"}
        if v not in allowed:
            raise ValueError(f"Category must be one of: {allowed}")
        return v

    class Config:
        extra = "forbid"    # reject unknown fields


class ProductRequestLenient(BaseModel):
    """
    Lenient validation — for importing legacy or messy data.
    Extra fields are IGNORED. Most fields optional with defaults.
    """
    name:            str            = Field(..., min_length=1)
    price:           float          = Field(..., gt=0)
    category:        Optional[str]  = Field(default="Unknown")
    image_path:      Optional[str]  = Field(default=None)
    additional_info: Optional[str]  = Field(default=None)
    target_audience: Optional[str]  = Field(default=None)

    class Config:
        extra = "ignore"    # silently drop unknown fields


# Test same product against both models
test_product = {
    "name": "Blue Shirt",
    "price": 29.99,
    "category": "Apparel",
    "image_path": "product_images/0.jpg",
    "mystery_field": "this field does not exist in the model",
}

print("\nTesting strict model:")
try:
    p_strict = ProductRequestStrict(**test_product)
    print(f"  ✓ Passed: {p_strict.name}")
except ValidationError as e:
    print(f"  ✗ Failed: {e.errors()[0]['msg']}")

print("\nTesting lenient model:")
try:
    p_lenient = ProductRequestLenient(**test_product)
    print(f"  ✓ Passed: {p_lenient.name} (extra field silently ignored)")
except ValidationError as e:
    print(f"  ✗ Failed: {e.errors()[0]['msg']}")

print("\nField constraint reference:")
constraints = [
    ("gt=0",           "value must be > 0"),
    ("ge=0",           "value must be >= 0"),
    ("lt=100",         "value must be < 100"),
    ("le=100",         "value must be <= 100"),
    ("min_length=2",   "string at least 2 chars"),
    ("max_length=50",  "string max 50 chars"),
    ("pattern=...",    "must match regex pattern"),
    ("extra='forbid'", "reject unknown fields"),
    ("extra='ignore'", "silently drop unknown fields"),
    ("extra='allow'",  "keep unknown fields as-is"),
]
for constraint, explanation in constraints:
    print(f"  {constraint:<20} → {explanation}")

print("\n✓ Step 5 complete")


STEP 5: VALIDATION STRICTNESS CONTROL

Testing strict model:
  ✗ Failed: Extra inputs are not permitted

Testing lenient model:
  ✓ Passed: Blue Shirt (extra field silently ignored)

Field constraint reference:
  gt=0                 → value must be > 0
  ge=0                 → value must be >= 0
  lt=100               → value must be < 100
  le=100               → value must be <= 100
  min_length=2         → string at least 2 chars
  max_length=50        → string max 50 chars
  pattern=...          → must match regex pattern
  extra='forbid'       → reject unknown fields
  extra='ignore'       → silently drop unknown fields
  extra='allow'        → keep unknown fields as-is

✓ Step 5 complete


C:\Users\eugnm\AppData\Local\Temp\ipykernel_23228\1743860651.py:21: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator("name")
C:\Users\eugnm\AppData\Local\Temp\ipykernel_23228\1743860651.py:27: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  @validator("category")
C:\Users\eugnm\AppData\Local\Temp\ipykernel_23228\1743860651.py:10: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict inst

In [5]:
# ============================================================
# STEP 6: INTEGRATING VALIDATED DATA WITH EXISTING CODE
# ============================================================

print("\n" + "=" * 50)
print("STEP 6: INTEGRATION WITH EXISTING CODE")
print("=" * 50)

# Create a validated product to work with
sample = ProductRequestLenient(
    name="Titan Women Silver Watch",
    price=89.99,
    category="Accessories",
    image_path="product_images/2.jpg",
    target_audience="women",
    additional_info="Water resistant, 2 year warranty"
)

print("\n1. Accessing fields as attributes:")
print(f"   product.name     = {sample.name}")
print(f"   product.price    = {sample.price}")
print(f"   product.category = {sample.category}")

print("\n2. Converting to dict (for code that expects dicts):")
product_dict = sample.dict()
print(f"   type: {type(product_dict)}")
print(f"   keys: {list(product_dict.keys())}")
print(f"   dict['name'] = {product_dict['name']}")

print("\n3. Converting to JSON string (for APIs and file writing):")
product_json = sample.json()
print(f"   type: {type(product_json)}")
print(f"   JSON: {product_json[:80]}...")

print("\n4. Passing validated data to ChatGPT:")

def generate_product_description(product: ProductRequestLenient) -> str:
    """
    Generate a marketing description using ChatGPT.
    Accepts a validated Pydantic model — guarantees clean data
    before spending API credits. Uses .dict() for easy prompt building.
    """
    p = product.dict()
    prompt = f"""Write a 2-sentence marketing description for:
Name: {p['name']}
Price: ${p['price']}
Category: {p['category']}
Info: {p.get('additional_info', 'N/A')}
Audience: {p.get('target_audience', 'general')}
Keep it punchy and professional."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
    )
    return response.choices[0].message.content.strip()

print("   Sending to ChatGPT...")
try:
    description = generate_product_description(sample)
    print(f"   ✓ Response: {description}")
except Exception as e:
    print(f"   ✗ API error: {e}")


print("\n5. Validating ChatGPT output with Pydantic:")

class ChatGPTProductResponse(BaseModel):
    """
    Validates structured output from ChatGPT.
    ChatGPT can hallucinate field names or wrong types —
    Pydantic catches this before it breaks downstream code.
    """
    product_name:    str            = Field(..., min_length=1)
    description:     str            = Field(..., min_length=10, max_length=500)
    suggested_price: Optional[float] = Field(default=None, gt=0)
    tags:            Optional[List[str]] = Field(default=None)

    class Config:
        extra = "ignore"


def get_structured_product_data(product_name: str):
    """Ask ChatGPT for structured JSON and validate the response."""
    prompt = f"""Return a JSON object for "{product_name}" with:
- product_name (string)
- description (1-2 sentences)
- suggested_price (number in USD)
- tags (list of 3 strings)
Return ONLY the JSON, no other text."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )
    raw = response.choices[0].message.content.strip()
    print(f"   Raw output: {raw[:100]}...")

    # Step 1: Parse JSON
    data, parse_error = safe_parse_json(raw)
    if parse_error:
        print(f"   ✗ Invalid JSON from ChatGPT: {parse_error}")
        return None

    # Step 2: Validate with Pydantic
    try:
        validated = ChatGPTProductResponse(**data)
        print(f"   ✓ Validated — {validated.product_name} | ${validated.suggested_price}")
        return validated.dict()
    except ValidationError as e:
        print(f"   ✗ Validation failed:")
        for err in e.errors():
            print(f"     '{err['loc'][0]}': {err['msg']}")
        return None

print("   Requesting structured data from ChatGPT...")
try:
    result = get_structured_product_data("Titan Women Silver Watch")
except Exception as e:
    print(f"   ✗ API error: {e}")

print("\n✓ Step 6 complete")
print("\n" + "=" * 50)
print("ALL STEPS COMPLETE")
print("=" * 50)


STEP 6: INTEGRATION WITH EXISTING CODE

1. Accessing fields as attributes:
   product.name     = Titan Women Silver Watch
   product.price    = 89.99
   product.category = Accessories

2. Converting to dict (for code that expects dicts):
   type: <class 'dict'>
   keys: ['name', 'price', 'category', 'image_path', 'additional_info', 'target_audience']
   dict['name'] = Titan Women Silver Watch

3. Converting to JSON string (for APIs and file writing):
   type: <class 'str'>
   JSON: {"name":"Titan Women Silver Watch","price":89.99,"category":"Accessories","image...

4. Passing validated data to ChatGPT:
   Sending to ChatGPT...
   ✗ API error: name 'client' is not defined

5. Validating ChatGPT output with Pydantic:
   Requesting structured data from ChatGPT...
   ✗ API error: name 'client' is not defined

✓ Step 6 complete

ALL STEPS COMPLETE


C:\Users\eugnm\AppData\Local\Temp\ipykernel_23228\1226238281.py:25: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  product_dict = sample.dict()
C:\Users\eugnm\AppData\Local\Temp\ipykernel_23228\1226238281.py:31: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  product_json = sample.json()
C:\Users\eugnm\AppData\Local\Temp\ipykernel_23228\1226238281.py:43: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  p = product.dict()
C:\Users\eugnm\AppData\Local\Temp\ipykernel_23228\1226238281